# Diffusion Model Training — Synthetic, Text-Free Forms

This notebook trains and tests the project's own PyTorch diffusion model. It does not use a pretrained diffusion model. Every generated image is fictional, visibly watermarked, **not a real prescription, and not for medical use**.

## Why retraining is needed
The existing checkpoint learned 64×64 grayscale layouts containing mostly a page border and a few sparse lines. Inspection showed that its architecture, grayscale `[-1, 1]` normalization, cosine schedule, velocity objective, and reverse-step order agree, while generated samples remain noisy. The limitation is the low-resolution, narrow procedural training distribution and limited optimization; a low validation velocity-MSE alone did not establish visual quality. This notebook uses richer geometric layouts at 128×128 and saves stage samples for inspection. It never draws text or uses real prescription images.

The RxHandBD-ML project dataset contains prescription images paired with handwritten transcription labels, so it is intentionally **not used for diffusion training**. The inspection cell reads only folder counts and CSV headers, not transcription contents or patient images.

## 1. Environment setup

Run this notebook from the project repository. In Colab, the setup cell clones the repository if its model files are not already available. A GPU is recommended; the 100-epoch 128-pixel run can be slow on CPU. Checkpoints and samples are written to a separate run directory so the existing Streamlit checkpoint is not overwritten.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

requirements = {
    "torch": "torch",
    "numpy": "numpy",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

candidates = [Path.cwd(), *Path.cwd().resolve().parents,
              Path("/content/AI_Prescription_Analyzer")]
PROJECT_ROOT = next((p for p in candidates
                     if (p / "models" / "diffusion_inference.py").is_file()), None)
if PROJECT_ROOT is None and IN_COLAB:
    subprocess.run([
        "git", "clone",
        "https://github.com/Shruti-25092005/AI-Medical-Prescription-Analyzer.git",
        "/content/AI_Prescription_Analyzer",
    ], check=True)
    PROJECT_ROOT = Path("/content/AI_Prescription_Analyzer")
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project, or clone the project repository first."
    )
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Persistent storage and training configuration

Colab users can mount Drive for checkpoints that survive a runtime reset. On local Jupyter the same cells use project folders. Edit the configuration below before the long training cell. `EPOCHS` is the target total for same-resolution resume; for a new 128-pixel run, it is 100 new epochs. The older 64-pixel weights can be warm-started, but the optimizer and loss history restart because resolution and training layouts changed.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/AI_Prescription_Analyzer/models")
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH = DRIVE_MODEL_DIR / "diffusion_model_128.pth"
    DRIVE_SOURCE = DRIVE_MODEL_DIR / "diffusion_model.pth"
    SAMPLE_DIR = Path("/content/drive/MyDrive/AI_Prescription_Analyzer/generated_images/diffusion_128")
else:
    CHECKPOINT_PATH = PROJECT_ROOT / "models" / "diffusion_model_128.pth"
    DRIVE_SOURCE = PROJECT_ROOT / "models" / "diffusion_model.pth"
    SAMPLE_DIR = PROJECT_ROOT / "generated_images" / "diffusion_128"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 100
TRAINING_SAMPLES = 512
BATCH_SIZE = 16 if DEVICE.type == "cuda" else 8
IMAGE_SIZE = 128
TIMESTEPS = 100
BASE_CHANNELS = 16
LEARNING_RATE = 2e-4
SEED = 42
CHECKPOINT_INTERVAL = 5
SAMPLE_INTERVAL = 10
SAMPLES_PER_STAGE = 2

from models.diffusion_inference import DEFAULT_CHECKPOINT_PATH
WARM_START_PATH = DRIVE_SOURCE if DRIVE_SOURCE.is_file() else DEFAULT_CHECKPOINT_PATH
if not WARM_START_PATH.is_file():
    WARM_START_PATH = None
print(f"Checkpoint output: {CHECKPOINT_PATH}")
print(f"Warm-start source: {WARM_START_PATH or 'none; training from scratch'}")
print(f"Training config: {EPOCHS} epochs, {TRAINING_SAMPLES} procedural forms, "
      f"batch {BATCH_SIZE}, {IMAGE_SIZE}x{IMAGE_SIZE}, {TIMESTEPS} timesteps")

## 3. Inspect the existing dataset without exposing prescription content

The current dataset is for handwriting recognition, not synthetic-image generation. This cell reports directory counts and column names only. Do not display or export the real prescription images or transcription strings in this notebook.

In [ ]:
import csv

DATASET_ROOT = PROJECT_ROOT / "dataset" / "RxHandBD-ML" / "RxHandBD-ML"
if DATASET_ROOT.is_dir():
    for folder_name in ("Train_Set", "Test_Set"):
        folder = DATASET_ROOT / folder_name
        print(f"{folder_name}: {sum(p.is_file() for p in folder.iterdir()) if folder.is_dir() else 0} files")
    for csv_path in sorted(DATASET_ROOT.glob("*Label.csv")):
        with csv_path.open(encoding="utf-8-sig", newline="") as stream:
            headers = next(csv.reader(stream), [])
        print(f"{csv_path.name}: columns={headers}")
    print("Excluded from diffusion training: real prescription image/transcription pairs.")
else:
    print("Prescription dataset not present in this checkout; procedural training is self-contained.")

## 4. Procedural dataset, visualization, and preprocessing

The synthetic training set consists only of randomized blank geometric layouts: a page frame, an abstract header panel, a grid/table, and a blank lower panel. There are no words, patient identifiers, medicine names, doses, or instructions. Preprocessing converts grayscale pixel values to float tensors in `[-1, 1]`, which is the same normalization used during diffusion training and inference.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from models.train_diffusion import SyntheticFormDataset
from models.diffusion_inference import render_synthetic_form

preview_images = [render_synthetic_form(i, IMAGE_SIZE, SEED) for i in range(6)]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for axis, image in zip(axes.flat, preview_images):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.axis("off")
fig.suptitle("Procedural training examples — no text or clinical content")
plt.tight_layout()
plt.show()

preview_tensor = SyntheticFormDataset(2, IMAGE_SIZE, SEED)[0]
print("Tensor shape:", tuple(preview_tensor.shape))
print("Normalized range:", float(preview_tensor.min()), float(preview_tensor.max()))
assert preview_tensor.shape == (1, IMAGE_SIZE, IMAGE_SIZE)
assert preview_tensor.min() >= -1 and preview_tensor.max() <= 1

## 5. Denoiser architecture and forward diffusion

The model is a residual U-Net with sinusoidal timestep embeddings. For cumulative alpha `ᾱₜ`, the forward process is `xₜ = √ᾱₜ x₀ + √(1−ᾱₜ) ε`. It predicts velocity `vₜ = √ᾱₜ ε − √(1−ᾱₜ) x₀`. These equations and the cosine schedule are shared with inference.

In [ ]:
from models.diffusion_inference import (
    DiffusionDenoiser,
    cosine_noise_schedule,
    velocity_to_clean_and_noise,
)

denoiser = DiffusionDenoiser(
    image_channels=1, base_channels=BASE_CHANNELS, timesteps=TIMESTEPS
).to(DEVICE)
clean_batch = preview_tensor.unsqueeze(0).to(DEVICE)
_, _, alpha_cumulative = cosine_noise_schedule(TIMESTEPS, DEVICE)
time_batch = torch.tensor([TIMESTEPS // 2], device=DEVICE)
noise = torch.randn_like(clean_batch)
alpha = alpha_cumulative[time_batch].view(-1, 1, 1, 1)
noisy_batch = alpha.sqrt() * clean_batch + (1 - alpha).sqrt() * noise
target_velocity = alpha.sqrt() * noise - (1 - alpha).sqrt() * clean_batch
predicted_velocity = denoiser(noisy_batch, time_batch)
print("Noisy input / target / prediction:", noisy_batch.shape,
      target_velocity.shape, predicted_velocity.shape)
recovered_clean, recovered_noise = velocity_to_clean_and_noise(
    noisy_batch, target_velocity, alpha.reshape(-1)
)
assert torch.allclose(recovered_clean, clean_batch, atol=1e-5)
assert torch.allclose(recovered_noise, noise, atol=1e-5)
print("Forward and velocity equations verified.")

## 6. Smoke test before full training

This short CPU/GPU test trains a tiny model in a temporary directory, reloads the checkpoint through the production loader, and generates a watermarked sample. It does not overwrite the existing or planned 128-pixel checkpoint. Passing this test verifies the pipeline, not image quality.

In [ ]:
import tempfile
from models.train_diffusion import train
from models.diffusion_inference import generate_images, load_diffusion_model

with tempfile.TemporaryDirectory() as temp_dir:
    smoke_checkpoint = Path(temp_dir) / "diffusion_smoke.pth"
    train(
        epochs=1, sample_count=4, batch_size=2, image_size=32,
        timesteps=8, base_channels=4, seed=7,
        checkpoint_path=smoke_checkpoint, sample_interval=0,
    )
    smoke_bundle = load_diffusion_model(smoke_checkpoint)
    smoke_images = generate_images(
        count=1, sampling_steps=8, seed=7, model_bundle=smoke_bundle
    )
    assert len(smoke_images) == 1
    assert smoke_images[0].size == (512, 512)
    assert smoke_images[0].getpixel((0, 0)) == (150, 25, 35)
    print("Smoke test passed: train -> save -> reload -> sample.")

## 7. Train, resume, and checkpoint

Checkpoints are atomically saved every five epochs. CSV training/validation velocity-MSE history is written each epoch. Two fully sampled, visibly watermarked examples and an epoch-labeled comparison grid are saved every ten epochs and at the final epoch. Same-resolution runs restore optimizer/EMA state when present. A 64-pixel checkpoint can initialize the new 128-pixel model weights, but its optimizer and old loss history are not reused.

In [ ]:
from models.train_diffusion import train
from models.diffusion_inference import CHECKPOINT_FORMAT_VERSION

resume_same_resolution = False
warm_start = WARM_START_PATH
if CHECKPOINT_PATH.is_file():
    existing = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    if existing.get("format_version") != CHECKPOINT_FORMAT_VERSION:
        raise ValueError("Output checkpoint has an incompatible format; choose a new output path.")
    if existing.get("image_size") == IMAGE_SIZE:
        already_trained = int(existing.get("training_epochs", 0))
        if already_trained >= EPOCHS:
            print(f"Checkpoint already has {already_trained} epochs. Increase EPOCHS to continue.")
        else:
            resume_same_resolution = True
            warm_start = None

ready_checkpoint = False
if CHECKPOINT_PATH.is_file():
    current_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    ready_checkpoint = (
        current_checkpoint.get("image_size") == IMAGE_SIZE
        and int(current_checkpoint.get("training_epochs", 0)) >= EPOCHS
    )
if not ready_checkpoint:
    train(
        epochs=EPOCHS,
        sample_count=TRAINING_SAMPLES,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        timesteps=TIMESTEPS,
        base_channels=BASE_CHANNELS,
        learning_rate=LEARNING_RATE,
        seed=SEED,
        checkpoint_path=CHECKPOINT_PATH,
        resume=resume_same_resolution,
        resume_from=warm_start if not resume_same_resolution else None,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        sample_interval=SAMPLE_INTERVAL,
        samples_per_checkpoint=SAMPLES_PER_STAGE,
        sample_directory=SAMPLE_DIR,
    )
else:
    print("Training skipped; configured checkpoint already meets EPOCHS.")

## 8. Inspect loss and compare training stages

Use both loss curves and actual samples to judge training. Falling velocity-MSE does not prove that images look good. Stop or revise the experiment if validation loss rises persistently, becomes non-finite, or the generated layouts remain noisy. Do not claim improved quality until the saved samples have been visually inspected.

In [ ]:
import pandas as pd

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
history = checkpoint.get("training_loss_history", [])
history_df = pd.DataFrame(history)
if history_df.empty:
    history_path = CHECKPOINT_PATH.with_name(f"{CHECKPOINT_PATH.stem}_training_history.csv")
    if history_path.is_file():
        history_df = pd.read_csv(history_path)
if not history_df.empty:
    display(history_df.tail())
    plt.figure(figsize=(9, 4))
    plt.plot(history_df["epoch"], history_df["train_velocity_mse"], label="train")
    plt.plot(history_df["epoch"], history_df["validation_velocity_mse"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Velocity MSE")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("No loss history is recorded in this checkpoint.")

comparison_grid = SAMPLE_DIR / "training_comparison_grid.png"
if comparison_grid.is_file():
    display(Image.open(comparison_grid))
else:
    print("No stage grid yet; wait for a configured sample interval or train to an interval.")

## 9. Load for inference and save synthetic output

Inference checks the checkpoint format, architecture, normalization, schedule, prediction type, and weight shapes. Sampling starts from random noise and traverses diffusion timesteps in reverse order using the same cosine schedule and velocity-prediction equations.

In [ ]:
from IPython.display import display
from PIL import Image
from models.diffusion_inference import SYNTHETIC_OUTPUT_LABEL

inference_bundle = load_diffusion_model(CHECKPOINT_PATH)
OUTPUT_DIR = SAMPLE_DIR / "inference_samples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
generated = generate_images(
    count=4,
    sampling_steps=inference_bundle["timesteps"],
    seed=SEED + 10_000,
    model_bundle=inference_bundle,
)
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for index, (axis, image) in enumerate(zip(axes.flat, generated), start=1):
    axis.imshow(image)
    axis.set_title(f"Synthetic sample {index}")
    axis.axis("off")
    image.save(OUTPUT_DIR / f"synthetic_form_{index:02d}.png")
plt.tight_layout()
plt.show()
print(SYNTHETIC_OUTPUT_LABEL)
print({key: inference_bundle[key] for key in
       ("image_size", "timesteps", "training_epochs", "training_sample_count", "validation_velocity_mse")})
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Generated files: {OUTPUT_DIR}")

## 10. Use the checkpoint in Streamlit

Keep model checkpoints out of Git. For local testing, run Streamlit with the candidate checkpoint path configured:

```powershell
$env:DIFFUSION_MODEL_PATH = "C:\path\to\diffusion_model_128.pth"
.\.venv\Scripts\python.exe -m streamlit run app.py
```

In Colab, download the trained checkpoint or store it in private model storage. Set `DIFFUSION_MODEL_PATH` to the mounted file, or configure `DIFFUSION_MODEL_ID`, `DIFFUSION_MODEL_FILENAME`, and `HF_TOKEN` through the deployment secret manager. Restart Streamlit after changing the checkpoint. Review generated samples before selecting the new checkpoint for deployment.